# Object Model, Descriptors, `__slots__` & `__new__`

Hooks that control how objects are created and how attributes are accessed.

| Tool | Purpose |
|---|---|
| `__new__(cls, ...)` | Create and return a new instance |
| Descriptors | Objects that define `__get__`, `__set__` or `__delete__` |
| `__set_name__` | Receive the attribute name when a descriptor is assigned in a class |
| `__getattribute__` | Called for every attribute access |
| `__getattr__` | Called only when normal lookup fails |
| `__setattr__` / `__delattr__` | Intercept setting and deleting attributes |
| `__slots__` | Declare a fixed set of instance attributes |
| `__class__` / `type(obj)` | Show an object's class |

---

## `__new__`

`__new__(cls, ...)` is responsible for creating and returning a new instance.

It is useful for advanced object creation patterns, such as:

- Immutable built-in subclasses
- Singleton-like patterns
- Controlling instance creation

Normal application code usually only needs `__init__`.

## Descriptors

A descriptor is an object implementing one or more of:

```text
__get__
__set__
__delete__
```

A descriptor assigned as a class attribute can control attribute access.

Descriptors are the machinery behind several Python features, including:

- Bound methods
- `property`
- `classmethod`
- `staticmethod`
- `cached_property`
- `super`
- Many framework APIs

## `__set_name__`

A descriptor can implement:

```python
__set_name__(owner, name)
```

Python calls it when the descriptor is assigned as part of class creation.

This lets a descriptor know the class and attribute name it was assigned to.

## `__getattribute__`

Called for attribute access:

```python
obj.name
```

It is very powerful and easy to misuse.

Usually prefer `__getattr__` for a fallback when an attribute was not found.

## `__getattr__`

Called only when normal attribute lookup fails.

This makes it useful for dynamic fallback behavior.

## `__setattr__` / `__delattr__`

These intercept setting and deleting attributes.

Be careful to avoid infinite recursion. `object.__setattr__` is often used internally.

## `__slots__`

`__slots__` lets a class declare a fixed set of instance attributes and can remove the normal per-instance `__dict__` unless one is included.

Possible benefits:

- Lower memory overhead for many instances
- Restricting arbitrary new instance attributes
- Supporting some specialized object layouts

Trade-offs:

- More restrictions
- Multiple inheritance can make slot design more complicated
- Some libraries/features expect `__dict__`

Use it for a reason, not automatically.

## `__class__` and `type(obj)`

Both can show the object's class:

```python
obj.__class__
type(obj)
```

`type(obj)` is generally clearer for ordinary type inspection.

## Source

- Python Data Model: https://docs.python.org/3/reference/datamodel.html
- Descriptor Guide: https://docs.python.org/3/howto/descriptor.html

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.private_name = f"_{name}"

    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError("Value must be positive.")
        setattr(obj, self.private_name, value)


class Product:
    price = Positive()

    def __init__(self, price):
        self.price = price


item = Product(50)
print(item.price)


class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


point = Point(10, 20)
print(point.x, point.y)

# point.z = 30  # AttributeError: no slot for "z"